In [1]:
import os
import rasterio
from rasterio.enums import Resampling
from tqdm import tqdm

In [2]:
# Đầu vào - đầu ra
SRC_32648 = r"E:\DownloadData\co2_ban_do\output_32648"      # đầu vào từ Bước 1
OUT_500M  = r"E:\DownloadData\co2_ban_do\output_500m"        # đầu ra Bước 2
os.makedirs(OUT_500M, exist_ok=True)

# ================================
# RESAMPLE 250m → 500m
# ================================
def resample_to_500m(src_path, dst_path):

    with rasterio.open(src_path) as src:

        # Lấy bounding box
        left, bottom, right, top = src.bounds

        # Tính kích thước mới
        new_width  = int((right - left) / 500)
        new_height = int((top - bottom) / 500)

        # Tạo transform mới
        new_transform = rasterio.transform.from_origin(left, top, 500, 500)

        # Copy metadata
        profile = src.profile.copy()
        profile.update({
            "width": new_width,
            "height": new_height,
            "transform": new_transform
        })

        # Đọc + resample
        data = src.read(
            out_shape=(src.count, new_height, new_width),
            resampling=Resampling.bilinear
        )

        # Ghi file
        with rasterio.open(dst_path, "w", **profile) as dst:

            # GIỮ LẠI TÊN BAND
            if src.descriptions:
                for i, desc in enumerate(src.descriptions):
                    dst.set_band_description(i + 1, desc)

            dst.write(data)


# ================================
# DUYỆT TẤT CẢ DATASET
# ================================
for dataset in os.listdir(SRC_32648):

    dataset_path = os.path.join(SRC_32648, dataset)
    if not os.path.isdir(dataset_path):
        continue

    print("\n==============================")
    print("📌 Dataset:", dataset)
    print("==============================")

    for year_folder in os.listdir(dataset_path):

        year_path = os.path.join(dataset_path, year_folder)
        if not os.path.isdir(year_path):
            continue

        out_year_path = os.path.join(OUT_500M, dataset, year_folder + "_500m")
        os.makedirs(out_year_path, exist_ok=True)

        tifs = [f for f in os.listdir(year_path) if f.endswith(".tif")]

        for fname in tqdm(tifs, desc=f"{dataset} {year_folder}"):

            src_file = os.path.join(year_path, fname)
            dst_file = os.path.join(out_year_path, fname.replace("_32648.tif", "_500m.tif"))

            resample_to_500m(src_file, dst_file)

print("\n🎉 DONE — RESAMPLE 250m → 500m HOÀN CHỈNH!")



📌 Dataset: chirps


chirps chirps_precipitation_2024: 100%|██████████| 31/31 [00:06<00:00,  4.46it/s]



📌 Dataset: era5


era5 era5_2024: 100%|██████████| 31/31 [01:53<00:00,  3.67s/it]



📌 Dataset: et_pet


...


📌 Dataset: par


par par_2024: 100%|██████████| 24/24 [00:21<00:00,  1.13it/s]



📌 Dataset: smap


smap smap_2024: 100%|██████████| 31/31 [00:07<00:00,  4.04it/s]


🎉 DONE — RESAMPLE 250m → 500m HOÀN CHỈNH!


In [2]:

# Bước 2: Resample từ 250m → 500m cho các raster đã chuyển CRS (32648)
SRC_FINE = "E://DownloadData//co2_ban_do//output_32648_soilgrids"
OUT_500_FINE = "E://DownloadData/co2_ban_do/output_500m_soilgrids"
os.makedirs(OUT_500_FINE, exist_ok=True)

TARGET_RES = 500  # 500m
TARGET_CRS = "EPSG:32648"

files = [f for f in os.listdir(SRC_FINE) if f.endswith(".tif")]

for fname in tqdm(files, desc="Resample 250m → 500m"):
    src_path = os.path.join(SRC_FINE, fname)
    out_path = os.path.join(OUT_500_FINE, fname.replace(".tif","_500m.tif"))

    with rasterio.open(src_path) as src:
        # Tính bounding box và kích thước 500m
        left, bottom, right, top = src.bounds
        new_width  = int((right  - left) / TARGET_RES)
        new_height = int((top    - bottom) / TARGET_RES)

        new_transform = rasterio.transform.from_origin(left, top, TARGET_RES, TARGET_RES)

        profile = src.profile.copy()
        profile.update({
            "height": new_height,
            "width": new_width,
            "transform": new_transform
        })

        data = src.read(
            out_shape=(src.count, new_height, new_width),
            resampling=Resampling.average   # quan trọng! aggregate → 500m
        )

        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(data)

print("\n🎉 XONG CELL 2 — FINE RASTER (250m) → 500m")


Resample 250m → 500m: 100%|██████████| 6/6 [00:05<00:00,  1.11it/s]


🎉 XONG CELL 2 — FINE RASTER (250m) → 500m
